In [1]:
import sys, os, json, torch

BACKEND_PATH = os.path.join(os.getcwd(), '..', 'backend')
if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

from app.models.nanogpt import NanoGPT

DATA_DIR    = os.path.join(os.getcwd(), '..', 'data', 'fr')
WEIGHTS_DIR = os.path.join(BACKEND_PATH, 'app', 'weights')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print('Setup OK ✓')

Setup OK ✓


In [2]:
# ── Mapping catégorie → token de contrôle ────────────────────
#  Format fichier  :  {type}_{secteur}.txt
#  Token généré   :  #{TYPE}#{SECTEUR}#

CATEGORIES = [
    ('marque',  'luxe'),
    ('marque',  'tech'),
    ('marque',  'food'),
    ('marque',  'general'),
    ('societe', 'tech'),
    ('societe', 'services'),
    ('societe', 'industrie'),
    ('societe', 'general'),
]

def load_category(type_, secteur):
    """Charge un fichier catégorie et préfixe chaque nom avec son token."""
    path = os.path.join(DATA_DIR, f'{type_}_{secteur}.txt')
    if not os.path.exists(path):
        print(f'  ⚠️  Fichier manquant : {path}')
        return []
    with open(path, encoding='utf-8') as f:
        names = [l.strip() for l in f if l.strip() and ' ' not in l.strip()]
    token = f'#{type_.upper()}#{secteur.upper()}#'
    return [f'{token}{name}' for name in names]

all_lines = []
for type_, secteur in CATEGORIES:
    lines = load_category(type_, secteur)
    all_lines.extend(lines)
    print(f'  {type_:8s} / {secteur:12s} → {len(lines):4d} noms  (ex: {lines[0] if lines else "vide"})')

# Le texte d'entraînement = tous les noms préfixés, séparés par \n
text = '\n'.join(all_lines) + '\n'
print(f'\nTotal : {len(all_lines)} noms  |  {len(text)} caractères')

  marque   / luxe         →  155 noms  (ex: #MARQUE#LUXE#ambria)
  marque   / tech         →  160 noms  (ex: #MARQUE#TECH#academix)
  marque   / food         →  118 noms  (ex: #MARQUE#FOOD#ambrosia)
  marque   / general      →  196 noms  (ex: #MARQUE#GENERAL#accessex)
  societe  / tech         →  207 noms  (ex: #SOCIETE#TECH#agencepro)
  societe  / services     →  215 noms  (ex: #SOCIETE#SERVICES#achatpro)
  societe  / industrie    →  163 noms  (ex: #SOCIETE#INDUSTRIE#agrex)
  societe  / general      →  120 noms  (ex: #SOCIETE#GENERAL#actifcore)

Total : 1334 noms  |  33436 caractères


In [3]:
# Le vocabulaire doit maintenant contenir '#' et les lettres majuscules
# car les tokens de contrôle utilisent '#MARQUE#LUXE#'

chars      = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Vérification : '#' doit être dans le vocab
assert '#' in stoi, '# absent du vocabulaire !'
print(f'Vocabulaire FR v2 : {vocab_size} caractères')
print(f'Présence du # : oui  →  index {stoi["#"]}')
print(f'Aperçu : {chars[:20]}...{chars[-10:]}')

# Sauvegarde du vocabulaire au format backend
vocab_pkg = {'stoi': stoi}
with open(os.path.join(WEIGHTS_DIR, 'vocab_fr.json'), 'w', encoding='utf-8') as f:
    json.dump(vocab_pkg, f, ensure_ascii=False, indent=2)
print('\nvocab_fr.json sauvegardé ✓')

Vocabulaire FR v2 : 48 caractères
Présence du # : oui  →  index 1
Aperçu : ['\n', '#', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'L', 'M', 'N', 'O', 'Q', 'R', 'S', 'T', 'U', 'V']...['r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'é']

vocab_fr.json sauvegardé ✓


In [4]:
# Encodage du texte complet en tenseur
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f'train_data : {len(train_data)} tokens')
print(f'val_data   : {len(val_data)}   tokens')

# Exemple décodé pour vérifier
sample = ''.join(itos[i.item()] for i in data[:80])
print(f'\nAperçu texte encodé/décodé :\n{sample}')

train_data : 30092 tokens
val_data   : 3344   tokens

Aperçu texte encodé/décodé :
#MARQUE#LUXE#ambria
#MARQUE#LUXE#auralis
#MARQUE#LUXE#aurelia
#MARQUE#LUXE#aurel


In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Périphérique : {device.upper()}')

# block_size augmenté à 32 pour absorber les tokens de contrôle
# Ex: '#MARQUE#LUXE#ambria' = 20 chars → block_size 32 suffit
BLOCK_SIZE = 32

model_fr = NanoGPT(
    vocab_size  = vocab_size,
    n_embed     = 64,
    n_head      = 4,
    n_layer     = 4,
    block_size  = BLOCK_SIZE,
    dropout     = 0.1
)
model_fr.to(device)
params = model_fr.count_params()
print(f'Paramètres : {params:,}  |  block_size : {BLOCK_SIZE}')

Périphérique : CPU
Paramètres : 207,232  |  block_size : 32


In [6]:
import torch.optim as optim

# ── Hyperparamètres ──────────────────────────────────────────
MAX_ITERS     = 5000   # augmenté vs Sprint3 (plus de données + tokens)
EVAL_INTERVAL = 500
BATCH_SIZE    = 64     # augmenté pour meilleure stabilité
LR            = 1e-3

optimizer = optim.AdamW(model_fr.parameters(), lr=LR)

def get_batch(split):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - BLOCK_SIZE, (BATCH_SIZE,))
    x  = torch.stack([d[i:i+BLOCK_SIZE]   for i in ix])
    y  = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

print('🚀 Entraînement FR v2 (avec tokens de contrôle)...')
model_fr.train()

for it in range(MAX_ITERS):
    if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
        model_fr.eval()
        with torch.no_grad():
            _, lt = model_fr(*get_batch('train'))
            _, lv = model_fr(*get_batch('val'))
        print(f'Iter {it:5d} | Train: {lt.item():.4f} | Val: {lv.item():.4f}')
        model_fr.train()
    xb, yb = get_batch('train')
    _, loss = model_fr(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print('\n✓ Entraînement terminé')

🚀 Entraînement FR v2 (avec tokens de contrôle)...
Iter     0 | Train: 3.8708 | Val: 3.8885
Iter   500 | Train: 0.6976 | Val: 1.0829
Iter  1000 | Train: 0.5063 | Val: 1.0552
Iter  1500 | Train: 0.4457 | Val: 1.0356
Iter  2000 | Train: 0.3978 | Val: 1.1270
Iter  2500 | Train: 0.3570 | Val: 1.1064
Iter  3000 | Train: 0.3462 | Val: 1.1174
Iter  3500 | Train: 0.3448 | Val: 1.1149
Iter  4000 | Train: 0.3107 | Val: 1.1267
Iter  4500 | Train: 0.3308 | Val: 1.1304
Iter  4999 | Train: 0.2945 | Val: 1.1170

✓ Entraînement terminé


In [7]:
import torch.nn.functional as F

def generate_with_token(model, stoi, itos, ctrl_token, n=10,
                        temperature=0.8, top_k=15):
    """
    Génère des noms conditionnés par un token de contrôle.
    ctrl_token ex: '#MARQUE#LUXE#'
    """
    model.eval()
    names = []
    pad   = stoi.get('\n', 0)

    for _ in range(n * 4):
        # Encoder le token de contrôle
        ctx_ids = [stoi.get(c, 0) for c in ctrl_token if c in stoi]
        # Remplir le contexte à block_size avec du padding à gauche
        ctx = torch.zeros(1, model.block_size, dtype=torch.long)
        for j, cid in enumerate(ctx_ids[-model.block_size:]):
            ctx[0, -(len(ctx_ids))+j] = cid
        ctx = ctx.to(device)

        chars = []
        with torch.no_grad():
            for _ in range(20):
                logits, _ = model(ctx)
                logits = logits[:, -1, :] / max(temperature, 1e-5)
                v, _   = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
                probs  = F.softmax(logits, dim=-1)
                ix     = torch.multinomial(probs, 1).item()
                ch     = itos.get(ix, '')
                if ch in ('\n', '#') or ch == '':
                    break
                chars.append(ch)
                new      = torch.roll(ctx, -1, dims=1).clone()
                new[0,-1] = ix
                ctx      = new

        nm = ''.join(chars)
        # Exclure les résidus de token de contrôle
        nm = nm.split('#')[-1].strip()
        if 3 <= len(nm) <= 15 and nm not in names:
            names.append(nm)
        if len(names) >= n:
            break

    return names[:n]

# ── Test sur chaque catégorie ─────────────────────────────────
tests = [
    '#MARQUE#LUXE#',
    '#MARQUE#TECH#',
    '#MARQUE#FOOD#',
    '#SOCIETE#SERVICES#',
    '#SOCIETE#TECH#',
]
for token in tests:
    noms = generate_with_token(model_fr, stoi, itos, token, n=6)
    print(f'{token:25s} → {noms}')

#MARQUE#LUXE#             → ['rafia', 'seralia', 'chrysova', 'gloworia', 'notura', 'platina']
#MARQUE#TECH#             → ['datanex', 'protech', 'codalix', 'numialis', 'omegatech', 'dataforg']
#MARQUE#FOOD#             → ['vinalis', 'cerealia', 'freshova', 'tastefoood', 'naturalia', 'primeova']
#SOCIETE#SERVICES#        → ['assuprof', 'immosmart', 'compbel', 'santefusion', 'acquisalis', 'capitprime']
#SOCIETE#TECH#            → ['datasol', 'infosyn', 'digitnova', 'digitalis', 'dataste', 'netpulse']


In [8]:
model_path = os.path.join(WEIGHTS_DIR, 'model_fr.pt')
torch.save(model_fr.to('cpu').state_dict(), model_path)
print(f'Poids sauvegardés → {model_path} ✓')

# Vérifier que les deux fichiers existent
for fname in ['model_fr.pt', 'vocab_fr.json']:
    fp = os.path.join(WEIGHTS_DIR, fname)
    size = os.path.getsize(fp) // 1024
    print(f'  {fname:20s}  {size} Ko')

Poids sauvegardés → c:\Users\Lenovo\Documents\ProjetVersion1\notebooks\..\backend\app\weights\model_fr.pt ✓
  model_fr.pt           842 Ko
  vocab_fr.json         0 Ko
